Setting up the environment

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("E-commerce") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "64") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024)) \
    .config("spark.memory.fraction", "0.7") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.files.maxPartitionBytes", "64MB") \
    .getOrCreate()



In [2]:
spark

loading customer data 

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
import random
import hashlib

In [4]:
customer_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_customers_dataset.csv", header=True, inferSchema=True)

customer_df.show(5)
customer_df.printSchema()



+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows

root
 |-- customer_id: string 

optimizing 

In [5]:
print(SparkSession._instantiatedSession)

In [6]:
order_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_orders_dataset.csv", header=True, inferSchema=True)
order_item_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_order_items_dataset.csv", header=True, inferSchema=True)
payment_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_order_payments_dataset.csv", header=True, inferSchema=True)
product_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_products_dataset.csv", header=True, inferSchema=True)
review_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_order_reviews_dataset.csv", header=True, inferSchema=True)
seller_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_sellers_dataset.csv", header=True, inferSchema=True)
product_category_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\product_category_name_translation.csv", header=True, inferSchema=True)
geolocation_df = spark.read.csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\olist_geolocation_dataset.csv", header=True, inferSchema=True)

In [7]:
order_df.show(5)
order_item_df.show(5)
payment_df.show(5)
product_df.show(5)
review_df.show(5)
seller_df.show(5)
product_category_df.show(5)
geolocation_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

# cheking null values in the dataset

In [8]:

null_counts = customer_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in customer_df.columns])
null_counts.show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [9]:
# List of dataframes to check for null values
dataframes = [order_df, order_item_df, payment_df, product_df, review_df, seller_df, product_category_df, geolocation_df]

# Check for null values in each dataframe
for df in dataframes:
    null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
    null_counts.show()
    

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|fr

# checking duplicate values 

In [10]:
# Check for duplicate rows in customer_df
duplicate_rows = customer_df.groupBy(customer_df.columns).count().filter(col("count") > 1)

# Show duplicate rows if any
duplicate_rows.show()

+-----------+------------------+------------------------+-------------+--------------+-----+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|count|
+-----------+------------------+------------------------+-------------+--------------+-----+
+-----------+------------------+------------------------+-------------+--------------+-----+



# removing duplicaes 

In [11]:
row_count = customer_df.count()
print(f"The number of rows in customer_df is: {row_count}")
customer_df = customer_df.dropDuplicates()
row_count = customer_df.count()
print(f"The number of rows in customer_df after removing duplicates is: {row_count}")

The number of rows in customer_df is: 99441
The number of rows in customer_df after removing duplicates is: 99441


In [12]:
# List of dataframes to process
dataframes = [order_df, order_item_df, payment_df, product_df, review_df, seller_df, product_category_df, geolocation_df]

# Iterate through each dataframe, count rows, drop duplicates, and count rows again
for df in dataframes:
    table_name = [name for name, value in globals().items() if value is df][0]  # Get the variable name of the dataframe
    row_count = df.count()
    print(f"The number of rows in {table_name} is: {row_count}")
    df = df.dropDuplicates()
    row_count = df.count()
    print(f"The number of rows in {table_name} after removing duplicates is: {row_count}")

The number of rows in order_df is: 99441
The number of rows in order_df after removing duplicates is: 99441
The number of rows in order_item_df is: 112650
The number of rows in order_item_df after removing duplicates is: 112650
The number of rows in payment_df is: 103886
The number of rows in payment_df after removing duplicates is: 103886
The number of rows in product_df is: 32951
The number of rows in product_df after removing duplicates is: 32951
The number of rows in review_df is: 104162
The number of rows in review_df after removing duplicates is: 104077
The number of rows in seller_df is: 3095
The number of rows in seller_df after removing duplicates is: 3095
The number of rows in product_category_df is: 71
The number of rows in product_category_df after removing duplicates is: 71
The number of rows in geolocation_df is: 1000163
The number of rows in geolocation_df after removing duplicates is: 738332


# Handle Missing Values

1. Drop missing Values ( for non - critical columns )

2. Fill missing values ( for numerical columns )

3. Impute Missing Values ( for continous data )

In [13]:
order_df = order_df.na.drop(subset=['order_id','customer_id','order_status'])
order_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [14]:
order_df = order_df.fillna({'order_delivered_carrier_date': '1970-01-01', 'order_delivered_customer_date': '1970-01-01', 'order_estimated_delivery_date': '1970-01-01'})
order_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [15]:


review_df = review_df.filter(col("order_id").isNotNull() & col("review_id").isNotNull())

review_df.show()


+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [16]:
recomendo_count = review_df.groupBy("review_comment_title").count()
recomendo_count.show()


+--------------------+-----+
|review_comment_title|count|
+--------------------+-----+
|              Nota 8|    1|
|             Ótimo! |    9|
|  Friso lateral onix|    1|
|      Entrega rápida|   38|
|Entrega muito ráp...|    1|
|            Boa faca|    1|
|   Produto excelente|    8|
|Loja Séria Entreg...|    1|
| 2018-05-22 13:13:15|    1|
|               Média|    1|
|   Ainda não recebi.|    1|
|Muito bom e rápido. |    1|
|                ?!? |    1|
| Recomendadissimoooo|    1|
| Melhor loja não há!|    1|
|            Enganada|    1|
|   Produto incorreto|    6|
|    Cola para Cílios|    1|
|         Satisfeito |    7|
|Loja confiável pa...|    1|
+--------------------+-----+
only showing top 20 rows



In [17]:
product_df = product_df.filter(col("product_category_name").isNotNull())


In [18]:
product_df = product_df.fillna({'product_weight_g': 0, 'product_length_cm': 0, 'product_height_cm': 0, 'product_width_cm': 0})
null_counts = product_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in product_df.columns])
null_counts.show()



+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                    0|                  0|                         0|                 0|               0|                0|                0|               0|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [19]:
seller_df.show()

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|        penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|        sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|         curitiba|          PR|

# impute missing values

In [20]:
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=['payment_value'],outputCols=['payment_value']).setStrategy("median")
payment_df = imputer.fit(payment_df).transform(payment_df)
payment_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [21]:
payments_df_with_null = payment_df.withColumn('payment_value',when(col('payment_value')!= 99.33,col('payment_value')).otherwise(lit(None)))

In [22]:


imputer = Imputer(inputCols=['payment_value'],outputCols=['payment_value']).setStrategy('median')


payment_df = imputer.fit(payments_df_with_null).transform(payments_df_with_null)

payment_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        100.0|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [23]:

review_df = review_df.withColumn("review_comment_title", when(col("review_comment_title").isNull(), "sem_recomendacao").otherwise(col("review_comment_title")))
review_df = review_df.withColumn("review_comment_message", when(col("review_comment_message").isNull(), "sem_recomendacao").otherwise(col("review_comment_message")))
review_df = review_df.withColumn("review_score", when(col("review_score").isNull(), '1').otherwise(col("review_score")))
review_df = review_df.withColumn("review_creation_date", when(col("review_creation_date").isNull(), "1970-01-01").otherwise(col("review_creation_date")))
review_df = review_df.withColumn("review_answer_timestamp", when(col("review_answer_timestamp").isNull(), "1970-01-01").otherwise(col("review_answer_timestamp")))

In [24]:
review_df.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|    sem_recomendacao|      sem_recomendacao| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|    sem_recomendacao|      sem_recomendacao| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|    sem_recomendacao|      sem_recomendacao| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|    sem_recomendacao|  Recebi bem antes ...| 2017-04-21 00:00:00|   

# Standerdizing values

In [25]:
order_df.printSchema()
order_df = order_df.withColumn('order_approved_at',to_date(col('order_approved_at'))) \
.withColumn('order_delivered_carrier_date',to_date(col('order_delivered_carrier_date'))) \
.withColumn('order_delivered_customer_date',to_date(col('order_delivered_customer_date'))) \
.withColumn('order_estimated_delivery_date',to_date(col('order_estimated_delivery_date')))
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: date (nullable = true)
 |-- order_delivered_carrier_date: date (nullable = true)
 |-- order_delivered_customer_date: date (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)



In [26]:
order_df.show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|       2017-10-02|                  2017-10-04|                   2017-10-10|                   2017-10-18|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|       2018-07-26|                  2018-07-26|                   2018-08-07|                   2018-08-13|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered

In [27]:
payment_df = payment_df.withColumn('payment_type',when(col('payment_type') == 'boleto','bank transfer')
                                     .when(col('payment_type') == 'credit_card','credit card')
                                     .when(col('payment_type') == 'debit_card','debit card')
                                     .otherwise('other'))



payment_df.show(30)

+--------------------+------------------+-------------+--------------------+-------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|
+--------------------+------------------+-------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1|  credit card|                   8|        100.0|
|a9810da82917af2d9...|                 1|  credit card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1|  credit card|                   1|        65.71|
|ba78997921bbcdc13...|                 1|  credit card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1|  credit card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1|  credit card|                   2|        96.12|
|771ee386b001f0620...|                 1|  credit card|                   1|        81.16|
|3d7239c394a212faa...|                 1|  credit card|                   3|        51.84|

In [28]:
customer_df =  customer_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



# data transformation

In [29]:
customer_df = customer_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [30]:
labels = ["Recommended", "Highly Recommended", "Not Recommended", "Poor Quality", "Average"]


In [31]:


# Number of labels
n_labels = len(labels)

# Create a new column with a random index from 0 to n_labels-1
review_df = review_df.withColumn("label_index", floor(rand() * n_labels).cast("int"))

# Replace title with randomly selected label using `when`
from functools import reduce


# Build a chain of .when conditions to map index to label
conditions = reduce(
    lambda acc, i: acc.when(col("label_index") == i, lit(labels[i])),
    range(n_labels),
    when(col("label_index") == 0, lit(labels[0]))
)

# Apply to the title column
review_df = review_df.withColumn("review_comment_title", conditions).drop("label_index")


In [32]:
review_df.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|             Average|      sem_recomendacao| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|     Not Recommended|      sem_recomendacao| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|             Average|      sem_recomendacao| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|        Poor Quality|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [33]:
product_df = product_df.join(
product_category_df,"product_category_name","left")
product_df = product_df.drop("product_category_name")
product_df = product_df.withColumnRenamed("product_category_name_english", "product_category_name")

In [34]:
product_df = product_df.filter(col("product_category_name").isNotNull())

In [35]:
order_df.show(5)
order_item_df.show(5)
customer_df.show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|       2017-10-02|                  2017-10-04|                   2017-10-10|                   2017-10-18|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|       2018-07-26|                  2018-07-26|                   2018-08-07|                   2018-08-13|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered

# advanced transformations


In [36]:

order_item_df.select('price').summary().show()
payment_df.select('payment_value').summary().show()



+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|            112650|
|   mean|120.65373901464986|
| stddev|183.63392805025975|
|    min|              0.85|
|    25%|              39.9|
|    50%|             74.99|
|    75%|             134.9|
|    max|            6735.0|
+-------+------------------+

+-------+------------------+
|summary|     payment_value|
+-------+------------------+
|  count|            103886|
|   mean|154.10049650577756|
| stddev|217.49403480919355|
|    min|               0.0|
|    25%|             56.79|
|    50%|             100.0|
|    75%|            171.79|
|    max|          13664.08|
+-------+------------------+



In [37]:
quantiles = order_item_df.approxQuantile('price',[0.01,0.99],0.0)
low_cutoff, high_cutoff = quantiles[0], quantiles[1]
print(f"Low cutoff: {low_cutoff}, High cutoff: {high_cutoff}") 

Low cutoff: 9.99, High cutoff: 890.0


In [38]:
order_item_df = order_item_df.filter((col('price') >= low_cutoff) & (col('price') <= high_cutoff))
order_item_df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [39]:
quantiles = payment_df.approxQuantile('payment_value',[0.01,0.99],0.0)
low_cutoff, high_cutoff = quantiles[0], quantiles[1]
print(f"Low cutoff: {low_cutoff}, High cutoff: {high_cutoff}") 


Low cutoff: 6.69, High cutoff: 1040.01


In [40]:
payment_df = payment_df.filter((col('payment_value')>= low_cutoff) & (col('payment_value') <= high_cutoff))
payment_df.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit card|                   8|        100.0|
|a9810da82917af2d9...|                 1| credit card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows



In [41]:
product_df.show(5)

+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category_name|
+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|1e9e8ef04dbcff454...|                 40|                       287|                 1|             225|               16|               10|              14|            perfumery|
|3aa071139cb16b67c...|                 44|                       276|                 1|            1000|               30|               18|              20|                  art|
|96bd76ec8810374ed...|                 46|                       250|                 1|       

In [42]:
product_df = product_df.withColumn(
    'product_size_category',
    when(col('product_weight_g') <500,'Small')
    .when(col('product_weight_g').between(500,2000),'Medium')
    .otherwise('Large')
)



In [43]:
product_df = product_df.dropDuplicates(["product_id"])

In [44]:
category_words = {
    "electronics": ["Smartwatch", "Headphones", "VR Headset", "Speaker", "Monitor"],
    "furniture": ["Chair", "Table", "Desk", "Sofa", "Bookshelf"],
    "toys": ["Puzzle", "Lego Set", "RC Car", "Doll", "Board Game"],
    "clothing": ["T-Shirt", "Jeans", "Dress", "Jacket", "Sweater"],
    "beauty": ["Lipstick", "Shampoo", "Face Cream", "Perfume"],
    "default": ["Item", "Gadget", "Product", "Thing"]
}

adjectives = ["Sleek", "Eco-Friendly", "Modern", "Vintage", "Deluxe", "Compact", "Premium", "Colorful"]


def generate_unique_name(product_id, category):
    random.seed(int(hashlib.sha256(str(product_id).encode()).hexdigest(), 16)) 
    noun_list = category_words.get(category.lower(), category_words["default"])
    adjective = random.choice(adjectives)
    noun = random.choice(noun_list)
    suffix = random.randint(1000, 9999)
    return f"{adjective} {noun} #{suffix}"

generate_name_udf = udf(generate_unique_name, StringType())



product_df = product_df.withColumn(
    "product_name",
    generate_name_udf(col("product_id"), col("product_category_name"))
)


In [45]:
product_df.show()

+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+---------------------+--------------------+
|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category_name|product_size_category|        product_name|
+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+---------------------+--------------------+
|00088930e925c41fd...|                 56|                       752|                 4|            1225|               55|               10|              26|                 auto|               Medium|Colorful Product ...|
|000b8f95fcb9e0096...|                 25|                       364|                 3|             550

In [46]:
order_with_details = order_df.join(order_item_df,'order_id','left')\
.join(payment_df,'order_id','left')\
.join(customer_df,'customer_id','left')
order_with_details.show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+-------------+--------------------+-------------+--------------------+------------------------+-------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|payment_sequential| payment_type|payment_installments|payment_value|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+----------------

In [47]:
order_with_total_value = order_with_details.groupBy('order_id')\
.agg(sum('payment_value').alias('total_order_value'))
order_with_total_value.show(5)

+--------------------+-----------------+
|            order_id|total_order_value|
+--------------------+-----------------+
|ccbabeb0b02433bd0...|             43.0|
|cadbb3657dac2dbbd...|           409.79|
|21c71f62d2554e1ad...|            74.16|
|d86359a45e3f6c6d8...|            124.7|
|d4ab8a264235be36f...|           215.15|
+--------------------+-----------------+
only showing top 5 rows



# some exploratory data analysis

customer distribbution by states 

In [48]:
customer_df.show(5)

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|41c8f4b5708697913...|fe3634ccefbcdb053...|                    7170|    guarulhos|            SP|
|5f8b4882b5a4ec7bf...|694cb45ff29b603ac...|                   24120|      niteroi|            RJ|
|1f6ba909178c5b0e5...|b1a049b952fdc19b6...|                    4938|    sao paulo|            SP|
|b0964d76975dbc174...|24a5c2b24a4467c37...|                   86990|     marialva|            PR|
|a1d79dad5150f0021...|f1a7919983d547880...|                   66055|        belem|            PA|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 5 rows



In [49]:
customer_state_distribution = customer_df.groupBy("customer_state").count().orderBy("count", ascending=False)
customer_state_distribution.show(5)

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
+--------------+-----+
only showing top 5 rows



order status disribution

In [50]:
order_df.show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|       2017-10-02|                  2017-10-04|                   2017-10-10|                   2017-10-18|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|       2018-07-26|                  2018-07-26|                   2018-08-07|                   2018-08-13|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered

In [51]:
order_status_count = order_df.groupBy("order_status").count().orderBy("count",ascending=False)
order_status_count.show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



cardpayent type distribution

In [52]:
payment_df.show(5)
payment_type = payment_df.groupBy("payment_type").count().orderBy("count",ascending=False)
payment_type.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit card|                   8|        100.0|
|a9810da82917af2d9...|                 1| credit card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows

+-------------+-----+
| payment_type|count|
+-------------+-----+
|  credit card|75485|
|bank transfer|19621|
|        other| 5188|
|   debit card| 1516|
+----------

tope selling products

In [53]:
order_item_df.show(5)
top_products = order_item_df.groupBy("product_id").agg(sum("price").alias("total_sales")).orderBy("total_sales", ascending=False)
top_products.show()


+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

average delivery time 

In [54]:
delivery_df = order_df.select("order_id", "order_delivered_carrier_date", "order_delivered_customer_date")
delivery_df.show()

+--------------------+----------------------------+-----------------------------+
|            order_id|order_delivered_carrier_date|order_delivered_customer_date|
+--------------------+----------------------------+-----------------------------+
|e481f51cbdc54678b...|                  2017-10-04|                   2017-10-10|
|53cdb2fc8bc7dce0b...|                  2018-07-26|                   2018-08-07|
|47770eb9100c2d0c4...|                  2018-08-08|                   2018-08-17|
|949d5b44dbf5de918...|                  2017-11-22|                   2017-12-02|
|ad21c59c0840e6cb8...|                  2018-02-14|                   2018-02-16|
|a4591c265e18cb1dc...|                  2017-07-11|                   2017-07-26|
|136cce7faa42fdb2c...|                  1970-01-01|                   1970-01-01|
|6514b8ad8028c9f2c...|                  2017-05-22|                   2017-05-26|
|76c6e866289321a7c...|                  2017-01-26|                   2017-02-02|
|e69bfb5eb88e0ed

In [55]:
delivery_time_df = delivery_df.withColumn("delivery_time_in_days",date_diff(col("order_delivered_customer_date"), col("order_delivered_carrier_date")))
delivery_time_df.orderBy("delivery_time_in_days",ascending=False).show()

+--------------------+----------------------------+-----------------------------+---------------------+
|            order_id|order_delivered_carrier_date|order_delivered_customer_date|delivery_time_in_days|
+--------------------+----------------------------+-----------------------------+---------------------+
|2aa91108853cecb43...|                  1970-01-01|                   2017-11-20|                17490|
|1b3190b2dfa9d789e...|                  2018-02-26|                   2018-09-19|                  205|
|ca07593549f1816d2...|                  2017-03-08|                   2017-09-19|                  195|
|285ab9426d6982034...|                  2017-03-09|                   2017-09-19|                  194|
|2fb597c2f772eca01...|                  2017-03-13|                   2017-09-19|                  190|
|440d0d17af552815d...|                  2017-03-15|                   2017-09-19|                  188|
|2d7561026d542c8db...|                  2017-03-16|             

# save the file

In [56]:


customer_df = customer_df.toPandas()
customer_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\customer_df.csv", index=False)
review_df = review_df.toPandas()
review_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\review_df.csv", index=False)
seller_df = seller_df.toPandas()
seller_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\seller_df.csv", index=False)
product_category_df = product_category_df.toPandas()
product_category_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\product_category_df.csv", index=False)
geolocation_df = geolocation_df.toPandas()
geolocation_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\geolocation_df.csv", index=False)
payment_pd = payment_df.toPandas()
payment_pd.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\payment_df.csv", index=False)
order_with_details_pd = order_with_details.toPandas()
order_with_details_pd.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\order_with_details.csv", index=False)
order_with_total_value_pd = order_with_total_value.toPandas()
order_with_total_value_pd.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\order_with_total_value.csv", index=False)
order_df = order_df.toPandas()
order_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\order_df.csv", index=False)
order_item_df = order_item_df.toPandas()
order_item_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\order_item_df.csv", index=False)
product_df = product_df.toPandas()
product_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\product_df.csv", index=False)

saving insights

In [57]:
delivery_time_df = delivery_time_df.toPandas()
delivery_time_df.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\delivery_time_df.csv", index=False)